# Single Image Classification Inference

This notebook loads a trained EfficientNet-B0 model once, then allows you to classify multiple images by specifying their paths.

Steps:
1. Run Cell 2 (setup) once.
2. Run Cell 3 (load model) once.
3. Modify the `IMAGE_PATH` in Cell 4 and run it to classify images.

In [5]:
import os
import json
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms

# Setup configuration
MODEL_PATH = r"C:\Users\ibf\Desktop\TFM\Nou projecte\TFM\Models\baseline_model_1500img.pth"
CLASS_NAMES_PATH = r"C:\Users\ibf\Desktop\TFM\Nou projecte\TFM\Models\class_names.json"

IMAGE_SIZE = (224, 224)
TOP_K = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [6]:
# Load model once
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")

if not os.path.exists(CLASS_NAMES_PATH):
    raise FileNotFoundError(f"Class names file not found: {CLASS_NAMES_PATH}")

with open(CLASS_NAMES_PATH, "r", encoding="utf-8") as f:
    class_names = json.load(f)

num_classes = len(class_names)
print(f"Loaded {num_classes} classes: {class_names}")

# Build the same architecture used during training
model = models.efficientnet_b0(weights=None)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(256, num_classes),
)

checkpoint = torch.load(MODEL_PATH, map_location=device)
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

# Define preprocessing to match the baseline notebook: crop the mobile screenshot borders first, then resize
cut = 120
mobile_border_crop = transforms.Lambda(lambda img: img.crop((0, cut, img.width, img.height - cut)))
preprocess = transforms.Compose([
    mobile_border_crop,
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Model loaded and ready for inference.")

Loaded 8 classes: ['Banner aplicación', 'Cierre aplicación', 'Error aplicativo', 'Error funcional', 'Error terminal', 'Indeterminado', 'Revisión circuito', 'Timeout']
Model loaded and ready for inference.


## Predict on Image

Modify `IMAGE_PATH` below and run this cell to classify an image. You can run this cell multiple times with different image paths without reloading the model.

In [7]:
# Specify the image path here
IMAGE_PATH = r"C:\Users\ibf\Desktop\TFM\Nou projecte\Data\Revisión circuito\20260511052945_100168.png"

# Predict
if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f"Image file not found: {IMAGE_PATH}")

img = Image.open(IMAGE_PATH).convert("RGB")
input_tensor = preprocess(img).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_tensor)
    probs = torch.softmax(logits, dim=1).squeeze(0)

topk_probs, topk_idxs = torch.topk(probs, k=min(TOP_K, len(class_names)))

pred_idx = int(topk_idxs[0].item())
pred_label = class_names[pred_idx]
pred_conf = float(topk_probs[0].item())

print(f"Image: {IMAGE_PATH}")
print(f"Predicted class: {pred_label}")
print(f"Confidence: {pred_conf:.4f}")

print("\nTop predictions:")
for rank, (p, idx) in enumerate(zip(topk_probs.tolist(), topk_idxs.tolist()), start=1):
    print(f"{rank}. {class_names[int(idx)]}: {p:.4f}")

Image: C:\Users\ibf\Desktop\TFM\Nou projecte\Data\Revisión circuito\20260511052945_100168.png
Predicted class: Revisión circuito
Confidence: 1.0000

Top predictions:
1. Revisión circuito: 1.0000
2. Error funcional: 0.0000
3. Indeterminado: 0.0000
